In [1]:
import torch
import torch.nn as nn
from torch import tensor
import torch.nn.functional as F

### Using Pytorch

In [12]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=1,
    batch_first=True,
    bias=False
)

attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # W_Q
        [0.1, 0.2],
        [0.3, 0.4],

        # W_K
        [0.2, 0.3],
        [0.4, 0.5],

        # W_V
        [0.3, 0.4],
        [0.5, 0.6],
    ]),
    
    "out_proj.weight": torch.tensor([
        [1.0, 1.1],
        [1.2, 1.3],
    ])
})

x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
        [0.5, 0.6],
        [0.6, 0.7],
        [0.7, 0.8],
    ]
])
output, weights = attention(
    query=x,
    key=x,
    value=x
)

seq_len = x.size(1)
causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool),
    diagonal=1
)
causal_mask

output, weights = attention(
    query=x,
    key=x,
    value=x,
    attn_mask=causal_mask
)
output, weights

(tensor([[[0.4880, 0.5800],
          [0.6838, 0.8127],
          [0.8154, 0.9691],
          [0.9327, 1.1086],
          [1.0461, 1.2433]]], grad_fn=<TransposeBackward0>),
 tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
          [0.4873, 0.5127, 0.0000, 0.0000, 0.0000],
          [0.3164, 0.3365, 0.3471, 0.0000, 0.0000],
          [0.2300, 0.2474, 0.2565, 0.2660, 0.0000],
          [0.1774, 0.1929, 0.2012, 0.2098, 0.2187]]], grad_fn=<MeanBackward1>))

### Manual

In [4]:
x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
        [0.5, 0.6],
        [0.6, 0.7],
        [0.7, 0.8],
    ]
])
x.shape

W_Q = torch.tensor([
    [0.1, 0.2],
    [0.3, 0.4],
])

W_K = torch.tensor([
    [0.2, 0.3],
    [0.4, 0.5],
])

W_V = torch.tensor([
    [0.3, 0.4],
    [0.5, 0.6],
])

W_O = torch.tensor([
    [1.0, 1.1],
    [1.2, 1.3],
])

Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T

score = Q @ K.transpose(-2, -1)

score_scaled = score / torch.sqrt(torch.tensor(2.0))

In [3]:
seq_len = x.size(1)
causal_mask = torch.triu(
    torch.ones(seq_len, seq_len, dtype=torch.bool),
    diagonal=1
)
causal_mask

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [5]:
score_scaled

tensor([[[0.0366, 0.0652, 0.0795, 0.0938, 0.1080],
         [0.0649, 0.1155, 0.1409, 0.1662, 0.1915],
         [0.0791, 0.1407, 0.1715, 0.2024, 0.2332],
         [0.0932, 0.1659, 0.2022, 0.2386, 0.2749],
         [0.1073, 0.1911, 0.2329, 0.2748, 0.3166]]])

In [6]:
score_scaled = score_scaled.masked_fill(
    causal_mask,
    float("-inf")
)

In [7]:
score_scaled

tensor([[[0.0366,   -inf,   -inf,   -inf,   -inf],
         [0.0649, 0.1155,   -inf,   -inf,   -inf],
         [0.0791, 0.1407, 0.1715,   -inf,   -inf],
         [0.0932, 0.1659, 0.2022, 0.2386,   -inf],
         [0.1073, 0.1911, 0.2329, 0.2748, 0.3166]]])

In [8]:
weights = F.softmax(score_scaled, dim=-1)

In [9]:
weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4873, 0.5127, 0.0000, 0.0000, 0.0000],
         [0.3164, 0.3365, 0.3471, 0.0000, 0.0000],
         [0.2300, 0.2474, 0.2565, 0.2660, 0.0000],
         [0.1774, 0.1929, 0.2012, 0.2098, 0.2187]]])

In [10]:
attn = weights @ V

output = attn @ W_O.T

In [11]:
output

tensor([[[0.4880, 0.5800],
         [0.6838, 0.8127],
         [0.8154, 0.9691],
         [0.9327, 1.1086],
         [1.0461, 1.2433]]])